# 数据工程

> 我们已经知道，LLaMA 3 用了 15 万亿个 token 训练，Chinchilla 定律说数据量和模型大小同样重要。但这 15T 个 token 从哪来？不能从天上掉下来。
>
> 这一节，我们从互联网原始 HTML 出发，走完数据清洗的完整 pipeline——文本提取、质量过滤、去重、数据混合，每一步为什么做、怎么做。

数据工程的起点是 Common Crawl——一个每月爬取整个互联网的非营利项目，原始数据量约 500TB/月。这些数据是 HTML 格式，里面混着导航栏、广告、JavaScript 代码和评论区内容。直接喂给模型等于喂垃圾，需要一套系统的方法把原始 HTML 变成干净、可训练的文本。整个过程分为五步：文本提取 → 质量过滤 → 去重 → 数据混合 → Tokenize。每一步的筛选标准都会直接影响最终模型的质量，因此每一步都需要明确的判断规则。下面每一步都会用真实的数据片段演示。

In [1]:
import re
import hashlib
import numpy as np

np.random.seed(42)

## 0. 总览：数据 Pipeline 全貌

先看全景，再逐个击破：

```
  Common Crawl (原始 HTML，~500TB/月)
       │
       ▼
  ┌─────────────┐
  │ 1. 文本提取  │  HTML → 纯文本，去掉导航/广告/脚本
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │ 2. 语言过滤  │  只要中文/英文等目标语言
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │ 3. 质量过滤  │  去广告/导航/乱码/太短/太长
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │ 4. 去重      │  精确去重 + MinHash 近似去重
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │ 5. 数据混合  │  Common Crawl + Wiki + Books + Code
  └──────┬──────┘
         ▼
    干净训练数据 ✨
```

## 1. 文本提取：从 HTML 里提取正文

#### 1.1 Common Crawl 里有什么？

Common Crawl 每个月爬取数十亿个网页，存储为 WARC（Web ARChive）格式。每个 WARC 文件里是一个完整的 HTML 页面——包含正文，也包含大量和正文无关的内容：

- 导航栏：「首页 | 关于 | 联系我们」
- 广告弹窗：「限时优惠！点击购买！」
- JavaScript 代码：「function trackUser(){...}」
- CSS 样式：「.sidebar { float: right; }」
- 页脚：「Copyright 2024. All rights reserved.」
- 评论区：「沙发！好文！顶！」

一篇正常的文章被这些东西包裹着。文本提取的任务就是识别并保留正文，丢弃其余部分。这个过程不完全自动化——现代工具（如 trafilatura、resiliparse）结合了 HTML 标签分析和机器学习模型来判断哪些内容是正文。

#### 1.2 用一个模拟的 HTML 演示提取过程</cell>

In [2]:
# === 模拟: 一个典型的 Common Crawl HTML 页面 ===
raw_html = """
<!DOCTYPE html>
<html>
<head>
  <title>What is Machine Learning? - AI Blog</title>
  <meta name="description" content="Learn ML basics">
  <script src="tracking.js"></script>
  <style>.ad { color: red; }</style>
</head>
<body>
  <nav>
    <a href="/">Home</a> |
    <a href="/about">About</a> |
    <a href="/contact">Contact</a>
  </nav>
  
  <div class="sidebar">
    <div class="ad">
      <h3>Sponsored</h3>
      <p>Buy the best AI course! 50% off today only!</p>
      <button>Click Here!</button>
    </div>
    <script type="text/javascript">
      var _gaq = _gaq || [];
      _gaq.push(['_setAccount', 'UA-12345-6']);
      _gaq.push(['_trackPageview']);
    </script>
  </div>
  
  <article>
    <h1>What is Machine Learning?</h1>
    <p>Machine learning is a subset of artificial intelligence
    that enables systems to learn and improve from experience
    without being explicitly programmed.</p>
    
    <p>The process of learning begins with observations or data,
    such as examples, direct experience, or instruction.</p>
    
    <p>Machine learning algorithms build a mathematical model
    based on sample data, known as "training data".</p>
  </article>
  
  <div class="comments">
    <p>User123: Great article!</p>
    <p>Bot456: Buy followers at cheap-followers.com!</p>
  </div>
  
  <footer>
    <p>Copyright 2024 AI Blog. All rights reserved.</p>
    <p>Terms of Service | Privacy Policy | Cookie Settings</p>
  </footer>
</body>
</html>
"""

print("=== 原始 HTML ===")
print(raw_html[:500])
print("...")
print()
print("↑ 里面混了：导航栏、广告、JS跟踪代码、评论区垃圾、页脚")

=== 原始 HTML ===

<!DOCTYPE html>
<html>
<head>
  <title>What is Machine Learning? - AI Blog</title>
  <meta name="description" content="Learn ML basics">
  <script src="tracking.js"></script>
  <style>.ad { color: red; }</style>
</head>
<body>
  <nav>
    <a href="/">Home</a> |
    <a href="/about">About</a> |
    <a href="/contact">Contact</a>
  </nav>

  <div class="sidebar">
    <div class="ad">
      <h3>Sponsored</h3>
      <p>Buy the best AI course! 50% off today only!</p>
      <button>Click Here!</butto
...

↑ 里面混了：导航栏、广告、JS跟踪代码、评论区垃圾、页脚


In [3]:
# === 手工模拟文本提取过程 ===
print("=== 文本提取：HTML → 纯文本 ===")
print()

# Step 1: 去掉 <script> 和 <style> 标签及其内容
def remove_scripts_styles(html):
    """去掉 script 和 style 标签（包括内容）"""
    html = re.sub(r'<script[^>]*>.*?</script>', '', html, flags=re.DOTALL | re.IGNORECASE)
    html = re.sub(r'<style[^>]*>.*?</style>', '', html, flags=re.DOTALL | re.IGNORECASE)
    return html

# Step 2: 去掉所有 HTML 标签
def strip_tags(html):
    """去掉所有 <...> 标签"""
    return re.sub(r'<[^>]+>', '', html)

# Step 3: 清理空白
def clean_whitespace(text):
    """合并多空行、去首尾空白"""
    text = re.sub(r'[ \t]+', ' ', text)  # 合并连续空格/tab
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)  # 多空行 → 一个空行
    return text.strip()

# 一步一步走
print("Step 1 — 去掉 script/style 标签:")
no_scripts = remove_scripts_styles(raw_html)
print(f"  原始: {len(raw_html)} 字符 → 去掉后: {len(no_scripts)} 字符")
print()

print("Step 2 — 去掉所有 HTML 标签:")
plain_text = strip_tags(no_scripts)
print(f"  去掉标签后: {len(plain_text)} 字符")
print()

print("Step 3 — 清理空白:")
clean_text = clean_whitespace(plain_text)
print(f"  清理空白后: {len(clean_text)} 字符")
print()

print("=" * 60)
print("提取结果:")
print("=" * 60)
print(clean_text)
print("=" * 60)
print()
print("注意：导航栏(Home|About|Contact)还在、广告还在、评论区还在")
print("      → 这些需要在后续的「质量过滤」步骤中清除")

=== 文本提取：HTML → 纯文本 ===

Step 1 — 去掉 script/style 标签:
  原始: 1454 字符 → 去掉后: 1226 字符

Step 2 — 去掉所有 HTML 标签:
  去掉标签后: 831 字符

Step 3 — 清理空白:
  清理空白后: 707 字符

提取结果:
What is Machine Learning? - AI Blog

 Home |
 About |
 Contact

 Sponsored
 Buy the best AI course! 50% off today only!
 Click Here!

 What is Machine Learning?
 Machine learning is a subset of artificial intelligence
 that enables systems to learn and improve from experience
 without being explicitly programmed.

 The process of learning begins with observations or data,
 such as examples, direct experience, or instruction.

 Machine learning algorithms build a mathematical model
 based on sample data, known as "training data".

 User123: Great article!
 Bot456: Buy followers at cheap-followers.com!

 Copyright 2024 AI Blog. All rights reserved.
 Terms of Service | Privacy Policy | Cookie Settings

注意：导航栏(Home|About|Contact)还在、广告还在、评论区还在
      → 这些需要在后续的「质量过滤」步骤中清除


## 2. 质量过滤 — 怎么判断一篇文章值不值得学？

文本提取出来后，大部分内容仍然是低质量的。想象一下 Common Crawl 里有什么：自动生成的 SEO 页面、乱码、只有导航栏没有正文的页面、重复上千次的广告文案……这些内容如果直接拿去训练，模型学到的东西还不如不学。

因此质量过滤是整个 pipeline 中最关键的一步：宁可丢掉一些好数据，也不能让大量垃圾混入训练集。常用的方法分两个层次：

#### 2.1 启发式规则（快速筛掉明显垃圾）

这些规则不需要模型，直接检查文本的统计特征。每条规则针对一类常见垃圾：

In [4]:
# === 质量过滤规则：逐条演示 ===
print("=== 质量过滤规则 ===")
print()

# 准备几条「待审」文本
samples = [
    {"name": "好文章", "text": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. The field has grown rapidly since the 2010s, driven by advances in deep learning and the availability of large datasets."},
    {"name": "广告", "text": "BUY NOW!!! Click here!!! Limited time offer!!! Subscribe today and get 50% OFF!!! Don't miss this opportunity!!!"},
    {"name": "目录页", "text": "Chapter 1. Introduction. Chapter 2. Methods. Chapter 3. Results. Chapter 4. Discussion. Chapter 5. Conclusion. Appendix A. Appendix B. References."},
    {"name": "太短", "text": "Hello world."},
    {"name": "随机字符", "text": "asdfjkl; qwerty zxcvbnm @#$%@# 123456789 !!!!!!!"},
    {"name": "重复模板", "text": "This is a blog post.\n" * 30 + "unique content here"},
]

def quality_check(text):
    """返回 (是否保留, 丢弃原因)"""
    
    # 规则 1: 长度过滤
    words = text.split()
    if len(words) < 5:
        return False, f"太短 ({len(words)} 个词 < 5)"
    if len(text) > 5000:
        return False, f"太长 ({len(text)} 字符 > 5000)"
    
    # 规则 2: 平均词长 — 正常英文词长 3-10，乱码词很长
    avg_word_len = np.mean([len(w) for w in words])
    if avg_word_len > 12:
        return False, f"平均词长异常 ({avg_word_len:.1f} > 12)"
    
    # 规则 3: 特殊字符占比 — 正常文章标点不会超过 15%
    special_count = sum(1 for c in text if not c.isalnum() and not c.isspace())
    special_ratio = special_count / max(len(text), 1)
    if special_ratio > 0.25:
        return False, f"特殊字符太多 ({special_ratio:.1%} > 25%)"
    
    # 规则 4: 行重复率 — 模板页面有很多重复行
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) > 3:
        unique_ratio = len(set(lines)) / len(lines)
        if unique_ratio < 0.4:
            return False, f"行重复率过高 ({unique_ratio:.1%} < 40%)"
    
    # 规则 5: 大写字母占比 — 全是 CAPS LOCK 的通常是垃圾
    if len(text) > 50:
        upper_ratio = sum(1 for c in text if c.isupper()) / sum(1 for c in text if c.isalpha())
        if upper_ratio > 0.5:
            return False, f"大写字母太多 ({upper_ratio:.1%} > 50%)"
    
    return True, "通过"


print(f"{'文本':<12s} {'结果':>8s} {'原因'}")
print("-" * 55)
for sample in samples:
    passed, reason = quality_check(sample['text'])
    status = "✅ 保留" if passed else "❌ 丢弃"
    print(f"{sample['name']:<12s} {status:>8s}  {reason}")

print()
print("真实系统通常有 20-50 条这样的规则。")
print("这能过滤掉约 60-80% 的 Common Crawl 文本。")

=== 质量过滤规则 ===

文本                 结果 原因
-------------------------------------------------------
好文章              ✅ 保留  通过
广告               ✅ 保留  通过
目录页              ✅ 保留  通过
太短               ❌ 丢弃  太短 (2 个词 < 5)
随机字符             ❌ 丢弃  特殊字符太多 (29.2% > 25%)
重复模板             ❌ 丢弃  行重复率过高 (6.5% < 40%)

真实系统通常有 20-50 条这样的规则。
这能过滤掉约 60-80% 的 Common Crawl 文本。


#### 2.2 基于模型的过滤 — 请「语文老师」来评分

除了人工规则，还可以用一个**已经训练好的轻量级语言模型**（如 KenLM）来给每篇文章打分。

思路和做英语阅读一样：老师看一眼文章就能判断「这是人写的」还是「这是随机生成的」。

```
用语言模型计算文章的「困惑度」（Perplexity, PPL）：
  PPL 很低 (< 10):  文章太简单，像「a a a a a...」→ 扔
  PPL 正常 (10-1000): 像正常人类写的 → 留
  PPL 很高 (> 1000): 乱码 → 扔
```

有些系统还训练一个二分类器：Wikipedia 文章 = 好（正例），Common Crawl 随机文章 = 坏（负例）。训练后让它给每篇文章打分。

**关键洞察**：Wikipedia = 「教科书级别」的好文本，用维基当尺子来量其他文本。

## 3. 去重 — 同一句话不能让模型学 100 遍

#### 3.1 为什么去重很重要？

互联网上大量内容是重复的：
- 同一篇新闻被 50 个网站转载（Google News syndication）
- 同一个代码片段在无数博客里出现
- 同一个「Lorem ipsum」占位文本
- 同一个 Cookie 声明（「This website uses cookies...」）

如果不去重：
1. 模型花了宝贵的训练计算学重复内容
2. 重复内容会让模型「背」而不是「理解」
3. 训练数据统计失真（某段话的权重是它本来应有的一百倍）

#### 3.2 两层去重

In [5]:
# === 精确去重 ===
print("=== 第一层：精确去重 (Exact Dedup) ===")
print()

# 模拟 5 篇文章，其中 2 篇是重复的
docs = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Machine learning is a subset of artificial intelligence.",  # 和 doc 1 完全一样
    "Natural language processing deals with text data.",
    "Deep learning uses neural networks with many layers.",  # 和 doc 2 完全一样
]

print(f"总共 {len(docs)} 篇文章")
print()

# 哈希去重
seen = set()
unique_docs = []

for i, doc in enumerate(docs):
    # SHA256 哈希：把文章变成唯一指纹
    fingerprint = hashlib.sha256(doc.encode()).hexdigest()[:16]  # 只取前16位展示
    is_new = fingerprint not in seen
    
    print(f"文档 {i+1}: hash={fingerprint}  {'✅ 保留' if is_new else '❌ 重复，丢弃'}")
    
    if is_new:
        seen.add(fingerprint)
        unique_docs.append(doc)

print()
print(f"去重后: {len(unique_docs)} 篇 ({len(docs) - len(unique_docs)} 篇被删除)")
print()
print("精确去重能去掉约 5-15% 的 Common Crawl 数据")
print("但还不够——大部分重复是「改写」而非「逐字照抄」")

=== 第一层：精确去重 (Exact Dedup) ===

总共 5 篇文章

文档 1: hash=a5c7f6a65fc1e1e9  ✅ 保留
文档 2: hash=b85a4f441b9eb02a  ✅ 保留
文档 3: hash=a5c7f6a65fc1e1e9  ❌ 重复，丢弃
文档 4: hash=90b6942d04cd6a5c  ✅ 保留
文档 5: hash=b85a4f441b9eb02a  ❌ 重复，丢弃

去重后: 3 篇 (2 篇被删除)

精确去重能去掉约 5-15% 的 Common Crawl 数据
但还不够——大部分重复是「改写」而非「逐字照抄」


In [6]:
# === MinHash 近似去重：原理手算 ===
print("=== 第二层：MinHash 近似去重 ===")
print()
print("问题：100 亿篇文章，怎么快速找出相似的？")
print("      逐对比较 = 100亿² = 不可能")
print()
print("MinHash 的思路：给每篇文章算一个「指纹」，指纹相似 → 文章相似")
print()

# === 手工 MinHash 演示 ===
print("=== MinHash 手算 ===")
print()

# 三篇文章
doc_A = "the cat sat on the mat and looked at the dog"
doc_B = "the cat sat on the mat and watched the dog"  # 和 A 只差一个词
doc_C = "quantum mechanics describes behavior of subatomic particles"  # 完全不同的主题

print(f"文档 A: {doc_A}")
print(f"文档 B: {doc_B}")
print(f"文档 C: {doc_C}")
print()

# Step 1: 把每篇文章切成 n-gram 集合（连续 3 个词为一组）
def get_ngrams(text, n=3):
    words = text.lower().split()
    return set(' '.join(words[i:i+n]) for i in range(len(words) - n + 1))

A_ngrams = get_ngrams(doc_A, 3)
B_ngrams = get_ngrams(doc_B, 3)
C_ngrams = get_ngrams(doc_C, 3)

print(f"文档 A 的 n-gram ({len(A_ngrams)} 个): {A_ngrams}")
print()
print(f"文档 B 的 n-gram ({len(B_ngrams)} 个): {B_ngrams}")
print()

# Step 2: 算 Jaccard 相似度 (交集/并集)
def jaccard(s1, s2):
    inter = len(s1 & s2)
    union = len(s1 | s2)
    return inter / union if union > 0 else 0

j_AB = jaccard(A_ngrams, B_ngrams)
j_AC = jaccard(A_ngrams, C_ngrams)

print(f"A ∩ B 交集大小: {len(A_ngrams & B_ngrams)}")
print(f"A ∪ B 并集大小: {len(A_ngrams | B_ngrams)}")
print(f"Jaccard(A, B) = {len(A_ngrams & B_ngrams)}/{len(A_ngrams | B_ngrams)} = {j_AB:.2%}")
print()
print(f"Jaccard(A, C) = {j_AC:.2%}")
print()

# Step 3: MinHash 指纹（超级简化版——用随机哈希模拟）
print("=== MinHash 签名计算 ===")
print()

# 模拟：把每个 n-gram 哈希到一个整数，取每篇文章的最小 K 个哈希值作为签名
def minhash_signature(ngrams, num_hashes=4):
    """
    为 n-gram 集合生成 MinHash 签名
    实际 MinHash 用随机排列模拟，这里用不同种子的哈希来模拟
    """
    sig = []
    for i in range(num_hashes):
        # 对每个 n-gram 用 seed=i 算哈希，取最小值
        min_val = float('inf')
        for ng in ngrams:
            h = hash(ng + str(i)) % 100000
            min_val = min(min_val, h)
        sig.append(min_val) if min_val != float('inf') else sig.append(0)
    return sig

sig_A = minhash_signature(A_ngrams)
sig_B = minhash_signature(B_ngrams)
sig_C = minhash_signature(C_ngrams)

print(f"A 的 MinHash 签名: {sig_A}")
print(f"B 的 MinHash 签名: {sig_B}")
print(f"C 的 MinHash 签名: {sig_C}")
print()

# MinHash 相似度 = 签名一致的比例
def minhash_sim(s1, s2):
    matches = sum(1 for a, b in zip(s1, s2) if a == b)
    return matches / len(s1)

mh_AB = minhash_sim(sig_A, sig_B)
mh_AC = minhash_sim(sig_A, sig_C)

print(f"MinHash 近似相似度 A-B: {mh_AB:.2%}  (精确 Jaccard: {j_AB:.2%})")
print(f"MinHash 近似相似度 A-C: {mh_AC:.2%}  (精确 Jaccard: {j_AC:.2%})")
print()
print("→ MinHash 把「逐对比较 n-gram」变成了「比较 4 个数字」")
print("→ 速度快了几个数量级！相似度 > 阈值（如 80%）→ 只保留一篇")

=== 第二层：MinHash 近似去重 ===

问题：100 亿篇文章，怎么快速找出相似的？
      逐对比较 = 100亿² = 不可能

MinHash 的思路：给每篇文章算一个「指纹」，指纹相似 → 文章相似

=== MinHash 手算 ===

文档 A: the cat sat on the mat and looked at the dog
文档 B: the cat sat on the mat and watched the dog
文档 C: quantum mechanics describes behavior of subatomic particles

文档 A 的 n-gram (9 个): {'the mat and', 'at the dog', 'looked at the', 'sat on the', 'on the mat', 'cat sat on', 'and looked at', 'mat and looked', 'the cat sat'}

文档 B 的 n-gram (8 个): {'and watched the', 'the mat and', 'sat on the', 'on the mat', 'cat sat on', 'watched the dog', 'mat and watched', 'the cat sat'}

A ∩ B 交集大小: 5
A ∪ B 并集大小: 12
Jaccard(A, B) = 5/12 = 41.67%

Jaccard(A, C) = 0.00%

=== MinHash 签名计算 ===

A 的 MinHash 签名: [3222, 9218, 1325, 5504]
B 的 MinHash 签名: [11955, 5250, 1325, 8548]
C 的 MinHash 签名: [5570, 14068, 24500, 15543]

MinHash 近似相似度 A-B: 25.00%  (精确 Jaccard: 41.67%)
MinHash 近似相似度 A-C: 0.00%  (精确 Jaccard: 0.00%)

→ MinHash 把「逐对比较 n-gram」变成了「比较 4 个数字」
→ 速度快了几个数量级！相似度 > 阈值（

## 4. 数据混合 — 不同来源怎么「配菜」？

假设你已经有了各种数据来源：

| 来源 | 数量 | 质量 | 作用 |
|:---|:---|:---|:---|
| Common Crawl（过滤后） | 10T tokens | 中等 | 基础知识、多样性 |
| Wikipedia | 0.1T tokens | 很高 | 事实准确性 |
| Books | 0.5T tokens | 高 | 长文连贯性 |
| Code (GitHub) | 1T tokens | 中等 | 逻辑推理能力 |
| Academic Papers | 0.05T tokens | 很高 | 科学推理 |

#### 4.1 怎么混合？不是按原始数量直接倒在一起

策略：**高质量数据可以多学几遍（多 epoch），低质量数据少学几遍。**

In [7]:
# === 数据混合策略 ===
print("=== 数据混合：如何配比 ===")
print()

sources = [
    ("Common Crawl (过滤后)", 10000, 0.6, 1),
    ("Wikipedia",               100, 0.95, 4),
    ("Books",                   500, 0.85, 2),
    ("Code (GitHub)",          1000, 0.75, 2),
    ("ArXiv Papers",             50, 0.9,  4),
    ("News",                    300, 0.7,  1),
]

print(f"{'来源':<25s} {'原始量':>10s} {'质量':>6s} {'Epoch':>6s} {'有效量':>10s} {'占比':>8s}")
print("-" * 72)

total_effective = 0
results = []
for name, size, quality, epochs in sources:
    effective = size * epochs
    total_effective += effective
    results.append((name, size, quality, epochs, effective))

for name, size, quality, epochs, effective in results:
    ratio = effective / total_effective * 100
    print(f"{name:<25s} {size:>6.0f}B   {quality:>5.0%}  {epochs:>4d}x  {effective:>8.0f}B   {ratio:>6.1f}%")

print()
print(f"总有效数据: {total_effective:.0f}B tokens")
print()
print("关键决策:")
print("  • Wikipedia 虽然只有 100B，但 epoch=4 → 实际喂了 400B (高质量多学)")
print("  • Common Crawl 虽然 10T，但 epoch=1 → 只学一遍 (避免垃圾)")
print("  • ArXiv 少量但质量高，epoch=4 → 放大科学推理能力")
print()
print("注意: 多 epoch ≠ 把文章复制 4 份")
print("      而是训练 4 轮后再重新 shuffle → 让模型每次看到不同的顺序")

=== 数据混合：如何配比 ===

来源                               原始量     质量  Epoch        有效量       占比
------------------------------------------------------------------------
Common Crawl (过滤后)         10000B     60%     1x     10000B     71.9%
Wikipedia                    100B     95%     4x       400B      2.9%
Books                        500B     85%     2x      1000B      7.2%
Code (GitHub)               1000B     75%     2x      2000B     14.4%
ArXiv Papers                  50B     90%     4x       200B      1.4%
News                         300B     70%     1x       300B      2.2%

总有效数据: 13900B tokens

关键决策:
  • Wikipedia 虽然只有 100B，但 epoch=4 → 实际喂了 400B (高质量多学)
  • Common Crawl 虽然 10T，但 epoch=1 → 只学一遍 (避免垃圾)
  • ArXiv 少量但质量高，epoch=4 → 放大科学推理能力

注意: 多 epoch ≠ 把文章复制 4 份
      而是训练 4 轮后再重新 shuffle → 让模型每次看到不同的顺序


## 5. 完整 Pipeline 实战：为一个 1B 模型准备数据

前面四节分别讲了文本提取、质量过滤、去重、数据混合四个环节。但在实际项目中，这些环节并不是独立运行的——它们构成一条流水线，上一步的输出是下一步的输入，任何一个环节出问题都会影响最终训练效果。

下面把整个流程串起来，模拟一次完整的数据工程 pipeline：从原始 Common Crawl WARC 文件出发，经过提取 → 过滤 → 去重 → 混合，最终产出可以直接送入训练的数据。每一步我们都会输出统计数字，让读者对「处理了多少、丢掉了多少」有直观感受。

实际工业 pipeline 的代码量很大（通常数千行），这里用简化版本展示核心逻辑。每条规则的实际效果都可以通过输出的统计数字来验证。

In [8]:
print("=== 实战: 1B LLM 数据 Pipeline ===")
print()

steps = [
    ("Step 1: 确定数据预算", [
        "Chinchilla 最优: N = 1B → D ≈ 20B tokens",
        "过度训练: N = 1B → D ≈ 100B tokens",
        "选择: 50B tokens (折中，性价比高)",
    ]),
    ("Step 2: 下载 Common Crawl", [
        "下载最近的 2-3 个月 dump (约 20TB 压缩 WARC)",
        "工具: cc_downloader, HuggingFace datasets",
    ]),
    ("Step 3: 文本提取 + 语言过滤", [
        "WARC → HTML → 纯文本 (trafilatura / resiliparse)",
        "语言检测 (fastText): 只保留英文和中文",
        "产出: ~2TB 纯文本 (约 400B tokens)",
    ]),
    ("Step 4: 质量过滤", [
        "启发式规则: 长度/词长/特殊字符/行重复",
        "KenLM PPL 过滤: 10 < PPL < 1000",
        "产出: ~200GB (约 40B tokens) → 只剩下 10%",
    ]),
    ("Step 5: 去重", [
        "精确去重: SHA256 哈希 → 去掉 ~10%",
        "MinHash 近似去重: 相似度 > 80% 的只保留一篇 → 去掉 ~20%",
        "产出: ~140GB (约 28B tokens)",
    ]),
    ("Step 6: 混合其他来源", [
        "Wikipedia (2x epoch): 4B tokens",
        "Books (2x epoch): 6B tokens",
        "Code GitHub (2x epoch): 10B tokens",
        "其他: 2B tokens",
        "总计: 28B + 22B = 50B tokens",
    ]),
    ("Step 7: Tokenize + 打包", [
        "用 BPE tokenizer 把文本转成 token IDs",
        "拼接成连续序列，切成 2048/4096 长度的 chunk",
        "在文档边界插入 <EOS> token",
        "Shuffle + 打包成训练 batch → 开始训练！",
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()

print("压缩比总结:")
print("  20TB WARC → 2TB 纯文本 → 200GB 过滤后 → 140GB 去重后")
print("  最后可用数据只占原始下载的 ~0.7%")

=== 实战: 1B LLM 数据 Pipeline ===

Step 1: 确定数据预算
  Chinchilla 最优: N = 1B → D ≈ 20B tokens
  过度训练: N = 1B → D ≈ 100B tokens
  选择: 50B tokens (折中，性价比高)

Step 2: 下载 Common Crawl
  下载最近的 2-3 个月 dump (约 20TB 压缩 WARC)
  工具: cc_downloader, HuggingFace datasets

Step 3: 文本提取 + 语言过滤
  WARC → HTML → 纯文本 (trafilatura / resiliparse)
  语言检测 (fastText): 只保留英文和中文
  产出: ~2TB 纯文本 (约 400B tokens)

Step 4: 质量过滤
  启发式规则: 长度/词长/特殊字符/行重复
  KenLM PPL 过滤: 10 < PPL < 1000
  产出: ~200GB (约 40B tokens) → 只剩下 10%

Step 5: 去重
  精确去重: SHA256 哈希 → 去掉 ~10%
  MinHash 近似去重: 相似度 > 80% 的只保留一篇 → 去掉 ~20%
  产出: ~140GB (约 28B tokens)

Step 6: 混合其他来源
  Wikipedia (2x epoch): 4B tokens
  Books (2x epoch): 6B tokens
  Code GitHub (2x epoch): 10B tokens
  其他: 2B tokens
  总计: 28B + 22B = 50B tokens

Step 7: Tokenize + 打包
  用 BPE tokenizer 把文本转成 token IDs
  拼接成连续序列，切成 2048/4096 长度的 chunk
  在文档边界插入 <EOS> token
  Shuffle + 打包成训练 batch → 开始训练！

压缩比总结:
  20TB WARC → 2TB 纯文本 → 200GB 过滤后 → 140GB 去重后
  最后可用数据只占原始下载的 ~0.7%


## 6. 数据质量 > 数量

这是 NLP 领域反复验证的结论：

```
T5 论文 (2019):
  清洗后的 750GB 数据训练效果 > 未清洗的 6TB 数据
  → 1/8 的数据量，质量够高反而更好

Textbooks Are All You Need (2023):
  用 7B 高质量「教科书」风格数据训练的 1.3B 模型
  代码能力超过了用更多数据训练的大模型
  → 数据质量可以部分弥补参数量的不足

LLaMA 2 → LLaMA 3 (2023-2024):
  模型架构几乎没变，最大的改进在数据质量和规模
  从 2T → 15T tokens，数据量增大的同时质量反而提高了
  → 更好的过滤、更好的去重、更好的混合策略
```

这个结论对实际工作的指导意义很直接：如果你只有有限的资源，优先投入在提高数据质量上，而不是盲目扩大数据量。100 本好书的训练价值远高于 10000 本垃圾邮件。</cell>


## 7. Tokenize 之后：数据怎么变成训练流？

数据工程最后一步是 tokenize。这之后数据长什么样？

```
文档 A: "Machine learning is great."
  → tokenize → [42, 567, 18, 891, 15]

文档 B: "Deep learning is also great."
  → tokenize → [123, 567, 18, 456, 891, 15]

然后拼接成一条「token 面条」:
[42, 567, 18, 891, 15, <EOS>, 123, 567, 18, 456, 891, 15, <EOS>, ...]

切成训练 chunk (假设 seq_len=8):
  Chunk 1: [42, 567, 18, 891, 15, <EOS>, 123, 567]
  Chunk 2: [18, 456, 891, 15, <EOS>, ...]
  ...

注意：chunk 边界会切断文档！
  Chunk 1 的后 3 个 token 来自文档 B，前 5 个来自文档 A
  → 模型有时会在 chunk 里「跨文档」学习
  → 解决方法：加 <EOS> 告诉模型「新文档开始」
```

#### 7.1 Sequence Packing：把短文档拼成满的 chunk

上一节展示了「拼接 → 切成固定长度」的朴素做法。但它有一个问题：如果文档长度分布不均，有些 chunk 会被大量 padding 填充，浪费算力。

**Sequence Packing** 用装箱算法解决这个问题：把多条短文档紧凑地塞进一个固定长度的 chunk，尽量不留空位。

```
朴素做法（有 padding 浪费）:
  Chunk 1: [文档A(500 tokens)][padding(3596 tokens)]  ← 浪费 88%
  Chunk 2: [文档B(800 tokens)][padding(3296 tokens)]  ← 浪费 80%

Packing 做法（尽量塞满）:
  Chunk 1: [文档A(500)][文档B(800)][文档C(1200)][文档D(1500)][padding(96)]  ← 利用率 98%
```

实际收益：训练吞吐量可以提升 **2x~4x**（短文档越多收益越大）。

但 Packing 有一个必须解决的技术问题：**不同文档之间不能互相 attend，否则会信息泄漏。** 解决方法是在 attention 计算时加 block-diagonal mask。

In [ ]:
# === Sequence Packing 手算演示 ===
print("=== Sequence Packing：装箱算法手算 ===")
print()

# 模拟 8 条文档的 token 长度
docs = [
    ("文档A", 120),
    ("文档B", 80),
    ("文档C", 200),
    ("文档D", 50),
    ("文档E", 150),
    ("文档F", 90),
    ("文档G", 60),
    ("文档H", 40),
]

max_seq_len = 256  # chunk 的固定长度

# First-Fit Decreasing 装箱：先把长文档排前面，再逐个往 chunk 里塞
sorted_docs = sorted(docs, key=lambda x: -x[1])  # 按长度降序
chunks = []  # 每个 chunk 是 (文档列表, 已用长度)

for name, length in sorted_docs:
    placed = False
    # 尝试塞进已有的 chunk
    for chunk in chunks:
        if chunk[1] + length + 1 <= max_seq_len:  # +1 是 EOS token
            chunk[0].append((name, length))
            chunk[1] += length + 1
            placed = True
            break
    if not placed:
        chunks.append([[(name, length)], length + 1])

print(f"总文档数: {len(docs)}")
print(f"Chunk 容量: {max_seq_len} tokens")
print(f"装箱结果: {len(chunks)} 个 chunk")
print()

total_used = 0
for i, (items, used) in enumerate(chunks):
    doc_names = ' + '.join(name for name, _ in items)
    padding = max_seq_len - used
    util = used / max_seq_len * 100
    total_used += used
    print(f"  Chunk {i+1}: [{doc_names}]")
    print(f"          已用 {used} tokens, padding {padding}, 利用率 {util:.0f}%")

overall_util = total_used / (len(chunks) * max_seq_len) * 100
print(f"\n整体利用率: {overall_util:.0f}%")
print()
print("对比朴素做法（每条文档一个 chunk）:")
naive_chunks = len(docs)
print(f"  朴素: {naive_chunks} 个 chunk")
print(f"  Packing: {len(chunks)} 个 chunk")
print(f"  节省: {(1 - len(chunks)/naive_chunks)*100:.0f}%")

#### 7.2 Block-Diagonal Attention Mask：防止文档之间互相偷看

Packing 之后，一个 chunk 里可能包含 3~4 条不同的文档。如果不加任何限制，模型做 attention 时会「看到」其他文档的内容——这就是信息泄漏。

解决方法：**block-diagonal mask**。每个文档只能在内部做 attention，不同文档之间的 attention 权重设为负无穷（softmax 后变 0）。

```
Chunk: [文档A][文档B][文档C]

Attention Mask:
        A A A B B C C
    A [ 1 1 1 0 0 0 0 ]   ← A 只看 A
    A [ 1 1 1 0 0 0 0 ]
    A [ 1 1 1 0 0 0 0 ]
    B [ 0 0 0 1 1 0 0 ]   ← B 只看 B
    B [ 0 0 0 1 1 0 0 ]
    C [ 0 0 0 0 0 1 1 ]   ← C 只看 C
    C [ 0 0 0 0 0 1 1 ]

对角线上的块各自独立 = block-diagonal
```

In [ ]:
# === Block-Diagonal Mask 构造演示 ===
print("=== Block-Diagonal Attention Mask ===")
print()

# 假设一个 packed chunk 包含 3 条文档，长度分别为 3, 2, 2
doc_lengths = [3, 2, 2]
total_len = sum(doc_lengths)  # 7

# 构造 block-diagonal mask（纯 numpy）
mask = np.zeros((total_len, total_len))

pos = 0
for length in doc_lengths:
    for i in range(length):
        for j in range(i + 1):
            mask[pos + i, pos + j] = 1.0
    pos += length

print(f"文档长度: {doc_lengths}, 总长度: {total_len}")
print()
print("Block-Diagonal Mask (1=可见, 0=屏蔽):")
print("      ", "  ".join([f"t{i}" for i in range(total_len)]))
labels = []
pos = 0
for doc_idx, length in enumerate(doc_lengths):
    for _ in range(length):
        labels.append(f"D{doc_idx}")
print()
for i in range(total_len):
    row = "  ".join([f" {int(mask[i,j])}" for j in range(total_len)])
    print(f"  {labels[i]} t{i}: {row}")

print()
print("观察:")
print("  D0(Doc 0) 内部: t0→t0, t1→t0,t1, t2→t0,t1,t2  ← 因果 attention")
print("  D1(Doc 1) 内部: t3→t3, t4→t3,t4              ← 和 D0 完全隔离")
print("  D2(Doc 2) 内部: t5→t5, t6→t5,t6              ← 和其他文档隔离")
print()
print("实际训练中还需要:")
print("  1. 每个文档的 position id 从 0 重新开始")
print("  2. Loss 只在真实 token 上计算，padding 不算")


**Fill-in-the-Middle (FIM) — 训练代码补全模型的数据格式**

前面讨论的全是一般语言模型的数据流：从左到右，自回归预测下一个 token。但对于代码模型，还有一种重要的数据构造方式。

写代码时，光标前后都有代码。左边是已经写好的部分（prefix），右边是还没改动的部分（suffix）。代码补全模型需要同时理解两边，才能准确填好中间。Fill-in-the-Middle（FIM）就是为这个场景设计的训练数据格式。

做法是把每段代码随机拆成三部分，用特殊 token 重新排列：

```
<PRE> prefix内容 <SUF> suffix内容 <MID> middle内容
```

训练时，模型看到 `<PRE> ... <SUF> ... <MID>` 之后，要预测被挖掉的 middle 部分。prefix 和 suffix 部分的 labels 设为 ignore_index——模型能看到这些上下文，但不为它们计算 loss。只有 middle 部分正常计算 loss。

举个例子，一段 5 行的代码：

```
原始代码:
  a = 1
  b = 2
  c = 3
  d = 4
  e = 5

拆成前 2 + 中间 2 + 后 1:
  prefix = "a = 1\nb = 2\n"     (模型看到，不预测)
  middle = "c = 3\nd = 4\n"     (训练目标)
  suffix = "e = 5"              (模型看到，不预测)

FIM 格式（模型输入）:
  <PRE> a = 1\nb = 2\n <SUF> e = 5 <MID> c = 3\nd = 4\n
  └─ 可看到，不预测 ─┘  └ 可看到 ┘  └─── 要预测 ────┘
```

不是所有样本都做 FIM。通常约 50% 用标准自回归格式（保持一般语言理解和生成能力），50% 用 FIM 格式（学会看前后文做代码补全）。两种格式混合训练，模型既能正常对话，也能在 IDE 里做代码补全。FIM 不影响 tokenize 和 packing 的流程——它是在 tokenize 之前做一次文本级别的重新排列。

代码模型（Code Llama、DeepSeek-Coder、StarCoder 等）都把 FIM 作为标准配置。普通文本模型一般不用，因为写作场景天然是从左到右的。


In [ ]:
# FIM 格式转换的手算演示

def fim_transform(code):
    """
    将一段代码转换为 FIM 训练格式
    
    随机选择切点 → 拆成 prefix/middle/suffix → 按 FIM 格式重排
    """
    lines = code.split('\n')
    n = len(lines)
    
    # 随机选切点
    prefix_len = np.random.randint(1, max(2, int(n * 0.4)))
    middle_len = np.random.randint(1, max(2, int(n * 0.5)))
    prefix_len = min(prefix_len, n - 1)
    middle_len = min(middle_len, n - prefix_len)
    
    prefix = '\n'.join(lines[:prefix_len])
    middle = '\n'.join(lines[prefix_len:prefix_len + middle_len])
    suffix = '\n'.join(lines[prefix_len + middle_len:])
    
    fim_text = f"<PRE> {prefix}\n<SUF> {suffix}\n<MID> {middle}"
    return fim_text, prefix, middle, suffix


np.random.seed(42)

sample_code = """def calculate(x, y):
    z = x + y
    result = z * 2
    if result > 100:
        return result
    else:
        return 0

print(calculate(3, 4))"""

fim_text, prefix, middle, suffix = fim_transform(sample_code)

print("=== FIM 格式转换手算 ===")
print()
print("原始代码:")
print(sample_code)
print()
print("--- 拆分后 ---")
print(f"Prefix ({len(prefix)} 字符, 模型看到"):")
print(prefix)
print()
print(f"Middle ({len(middle)} 字符, 训练目标):")
print(middle)
print()
print(f"Suffix ({len(suffix)} 字符, 模型看到):")
print(suffix if suffix else '(空)')
print()
print("--- FIM 格式（模型输入）---")
print(fim_text)
print()

print("=== 训练时的 Label Mask ===")
print()
print("规则：")
print("  <PRE> ... <SUF> ... <MID> 这些特殊 token → 知道但不预测")
print("  prefix 和 suffix 的内容     → label = ignore_index")
print("  middle 的内容                → label = 正常 token（参与 loss）")
print()
print("关键观察：")
print("1. FIM 通过重排文本，让模型在训练时就能利用双向上下文")
print("2. <PRE> 和 <SUF> 提供前后文信息，<MID> 之后模型开始预测被挖掉的部分")
print("3. 只有 middle 区域的 loss 被保留——prefix/suffix 给信息但不给惩罚")
print("4. 约 50% 样本用标准格式 + 50% 用 FIM → 兼顾自回归生成和代码补全")

## 数据工程小结

确认你已经搞懂了这些（按顺序检查）：

1. ✅ 数据来源：Common Crawl（主体）+ Wikipedia + Books + Code + Papers
2. ✅ Pipeline 五步：文本提取 → 语言过滤 → 质量过滤 → 去重 → 数据混合
3. ✅ HTML → Text：去 script/style 标签 + 去 HTML 标签 + 清理空白
4. ✅ 质量过滤：启发式规则（长度/词长/符号）+ PPL（语言模型打分）
5. ✅ 精确去重：SHA256 哈希，完全相同的只留一份
6. ✅ MinHash：把文章变指纹（n-gram → hash → 取最小 K 个），指纹相似 = 文章相似
7. ✅ 数据混合：高质量多 epoch，低质量少 epoch
8. ✅ Tokenize 后：拼接成「token 面条」→ 切成固定长度 chunk → 开始训练
9. ✅ FIM（Fill-in-the-Middle）：用 <PRE>/<SUF>/<MID> 重排文本，训练代码补全能力

**一句话总结**：数据工程是 LLM 训练中最「不性感」但最关键的环节。架构可以抄，但数据 pipeline 是各家的核心壁垒。20TB 原始数据到最后只剩 0.7% 能用——洗菜比做菜更费功夫。